# Task 4 — Goalkeeper Age Profile

## Analytic question formulation

**What is the average age of goalkeepers who appeared in at least one match
at the 2026 FIFA World Cup, and is it significantly different from 27 years
— the commonly cited "prime age" benchmark for outfield players in elite
football?**

Goalkeepers are widely believed to peak later and retain their place well
into their thirties, unlike outfield players. This task tests that belief
directly against a fixed benchmark (one-sample *t*-test), and additionally
cross-checks it against the observed age of outfield players from the same
tournament (two-sample comparison) who made at least one appearance.

**Skills demonstrated:** data wrangling → sampling → descriptive statistics
→ confidence interval → one-sample *t*-test (plus a supporting two-sample
comparison).

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data_prep import load_players
from stats_utils import describe, ci_mean, one_sample_ttest, two_sample_ttest, levene_test

np.random.seed(42)
pd.set_option("display.max_columns", None)

players = load_players()
players.head()

## Data wrangling

We reuse `players_raw.csv` via `data_prep.load_players()`, which already
restricts the data to players with `appearances >= 1`. Here the variable of
interest is simply `age` (in completed years at the start of the
tournament), split by position group (`GK` vs. outfield `DF`/`MF`/`FW`).

In [ ]:
goalkeepers = players[players["position"] == "GK"].copy()
outfield = players[players["position"].isin(["DF", "MF", "FW"])].copy()
print("Goalkeepers with >=1 appearance:", goalkeepers.shape[0])
print("Outfield players with >=1 appearance:", outfield.shape[0])
goalkeepers[["player_name", "team", "age", "appearances"]].sample(8, random_state=1)

## Data preparation and sampling

**Population:** every goalkeeper, and separately every outfield player,
who made at least one appearance at the 2026 World Cup.

**Variable of interest:** `age` (completed years at tournament start), one
observation per player.

**Sampling technique:** we draw a **simple random sample, without
replacement, of up to 40 players** from each population (goalkeepers /
outfield players), using a fixed random seed, again to mimic realistic
partial data availability and to satisfy the *n* ≥ 30 Central Limit
Theorem guideline for both the one-sample and two-sample tests below.

In [ ]:
def srs(df, n, seed=42):
    n = min(n, len(df))
    return df.sample(n=n, random_state=seed, replace=False)

gk_sample = srs(goalkeepers, 40)
outfield_sample = srs(outfield, 40)

print(f"Goalkeeper population: {len(goalkeepers)}  -> sample n={len(gk_sample)}")
print(f"Outfield population:   {len(outfield)}  -> sample n={len(outfield_sample)}")

## Descriptive statistics

In [ ]:
desc_table = pd.DataFrame(
    {"Goalkeepers": describe(gk_sample["age"]), "Outfield players": describe(outfield_sample["age"])}
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([gk_sample["age"], outfield_sample["age"]], tick_labels=["Goalkeepers", "Outfield"])
ax.axhline(27, color="red", linestyle="--", label="Benchmark: 27 years")
ax.set_ylabel("Age (years)")
ax.set_title("Player age by position group (sample)")
ax.legend()
plt.tight_layout()
plt.savefig("../report/figs/task4_box.png", dpi=120)
plt.show()

desc_table

## Inferential statistics — confidence interval

We estimate a 95% confidence interval for the **population mean age of
goalkeepers** who appeared at the tournament, using the *t*-distribution.

In [ ]:
ci = ci_mean(gk_sample["age"], confidence=0.95)
print(f"n = {ci['n']}")
print(f"Sample mean GK age = {ci['mean']:.2f} years")
print(f"95% CI: ({ci['ci_low']:.2f}, {ci['ci_high']:.2f}) years")
ci

## Inferential statistics — one-sample *t*-test

$H_0: \mu_{GK} = 27$

$H_1: \mu_{GK} \neq 27$

We test the goalkeeper sample mean age against the fixed benchmark of 27
years at $\alpha = 0.05$, then supplement with a two-sample comparison
against outfield players.

In [ ]:
result = one_sample_ttest(gk_sample["age"], popmean=27, alternative="two-sided")
for k, v in result.items():
    print(f"{k}: {v}")

alpha = 0.05
if result["p_value"] < alpha:
    print(f"\nReject H0 (p={result['p_value']:.4f} < {alpha}): GK mean age differs significantly from 27.")
else:
    print(f"\nFail to reject H0 (p={result['p_value']:.4f} >= {alpha}): no significant difference detected.")

print("\n--- Supporting two-sample comparison: goalkeepers vs. outfield players ---")
lev2 = levene_test(gk_sample["age"], outfield_sample["age"])
result2 = two_sample_ttest(gk_sample["age"], outfield_sample["age"], equal_var=lev2["p_value"] > 0.05)
for k, v in result2.items():
    print(f"{k}: {v}")

## Conclusion

*(Auto-filled after running the cells above with real data — summarize
whether goalkeeper mean age differs significantly from 27 at α=0.05, the
interpretation of the GK mean-age CI, and how it compares with outfield
players.)*